# Proof of concept — Case of Neodymium

This notebook calculates the neodymium need based on the pathway model

Make sure you are running this notebook from the repository root (or that the repo root is on your Python path), and that the `shared` package is installed:
```bash
pip install -e .
```

In [1]:
from shared.utils import run_pathway
import pandas as pd
import plotly.express as px
import os

In [2]:
results = run_pathway('run_poc')

[run_pathway] Window 1/1 done in 188.3s
[run_pathway] Total time: 188.3s


In [ ]:
#save_dir = os.path.expanduser('~/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/PoC')
save_dir = os.path.expanduser('~/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/PoC')
os.makedirs(save_dir, exist_ok=True)

In [ ]:
technologies = ['WIND_ONSHORE', 'PV_ROOF', 'NEW_HYDRO_DAM']
periods = ['2020_2025', '2025_2030', '2030_2035', '2035_2040', '2040_2045', '2045_2050']

df_plot = pd.DataFrame(
    {period: results['F_new'].loc[period].loc[technologies].squeeze() for period in periods},
    index=technologies
)

df_melted = df_plot.T.reset_index().rename(columns={'index': 'Période'}).melt(
    id_vars='Période', var_name='Technologies', value_name='Capacité'
)

fig = px.bar(df_melted, x='Période', y='Capacité', color='Technologies', barmode='stack')
fig.update_layout(xaxis_title='Période', yaxis_title='Capacité [GW]')
fig.show()

#filepath = os.path.join(save_dir, 'new_elec_tech.png')
#fig.write_image(filepath)

In [ ]:
# Charger les intensités
df_mi = pd.read_excel('/Users/Paolo/Documents/PdM_code/Material_intensities.xlsx', sheet_name='MI_Energy', index_col=0)
df_ms = pd.read_excel('/Users/Paolo/Documents/PdM_code/Material_intensities.xlsx', sheet_name='MS_Energy_Disag')

In [6]:
tech_mapping = {
    'WIND_ONSHORE': 'Wind_GB-DFIG_SCIG_Onshore',
    'PV_ROOF':      'Sol_C-si_Silver',
    'NEW_HYDRO_DAM':'Hydro'
    }
tech_groups = {
    'WIND_ONSHORE': {
        'iam_source': 'Capacity..Electricity..Wind..Onshore',
        'excel_techs': ['Wind_DD-EESG_Onshore', 'Wind_GB-DFIG_SCIG_Onshore', 'Wind_DD-PMSG_Onshore', 'Wind_GB-PMSG_Onshore']
    },
    'PV_ROOF': {
        'iam_source': 'Capacity..Electricity..Solar..PV',
        'excel_techs': ['Sol_C-si_Silver', 'Sol_C-si_Copper', 'Sol_CdTe', 'Sol_CIGS', 'Sol_a-SiGe']
    },
    'NEW_HYDRO_DAM': {
        'iam_source': None,
        'excel_techs': ['Hydro']
    }
}
period_to_decades = {
    '2020_2025': (2020, None),
    '2025_2030': (2020, 2030),
    '2030_2035': (2030, None),
    '2035_2040': (2030, 2040),
    '2040_2045': (2040, None),
    '2045_2050': (2040, 2050),
}

In [7]:
def get_ms(iam_source, period):
    d1, d2 = period_to_decades[period]
    if iam_source is None:
        return pd.Series({'Hydro': 1.0})
    ms = df_ms[df_ms['IAM_Energy_Sources'] == iam_source].set_index('Decade').drop(columns=['IAM_Energy_Sources'])
    ms = ms.apply(pd.to_numeric, errors='coerce')
    if d2 is None:
        return ms.loc[d1]
    return (ms.loc[d1] + ms.loc[d2]) / 2

In [ ]:
def plot_material_demand(material, cumulative=False, market_share=False, show = False):

    data = {}
    for period in periods:
        data[period] = {}
        if market_share:
            for model_tech, info in tech_groups.items():
                f_new = results['F_new'].loc[period].loc[model_tech].squeeze()
                ms = get_ms(info['iam_source'], period)
                for excel_tech in info['excel_techs']:
                    data[period][excel_tech] = f_new * ms[excel_tech] * df_mi.loc[material, excel_tech]
        else:
            for tech, excel_col in tech_mapping.items():
                data[period][tech] = results['F_new'].loc[period].loc[tech].squeeze() * df_mi.loc[material, excel_col]

    df_plot = pd.DataFrame(data)

    if cumulative:
        df_cumulative = df_plot.sum().cumsum()
        fig = px.line(x=periods, y=df_cumulative.values,
                      title=f'Demande cumulative en {material}',
                      labels={'x': 'Période', 'y': 'Demande cumulative [t]'})
    else:
        df_melted = df_plot.T.reset_index().rename(columns={'index': 'Période'}).melt(
            id_vars='Période', var_name='Technologies', value_name='Demande [t]'
        )
        fig = px.bar(df_melted, x='Période', y='Demande [t]', color='Technologies', barmode='stack',
                     title=f'Demande en {material}')
        fig.update_layout(xaxis_title='Période', yaxis_title='Demande [t]')
    if show:
        fig.show()
        if (cumulative and market_share):
            filepath = os.path.join(save_dir, 'neo_demand_cumul_ms.png')
        if (cumulative == False and market_share == False):
            filepath = os.path.join(save_dir, 'neo_demand_no_cumul_no_ms.png')
        if (cumulative == True and market_share == False):
            filepath = os.path.join(save_dir, 'neo_demand_cumul_no_ms.png')
        if (cumulative == False and market_share):
            filepath = os.path.join(save_dir, 'neo_demand_no_cumul_ms.png')
        #fig.write_image(filepath)

    return df_plot

# Exemple
plot_material_demand('Neodymium', show = True)
plot_material_demand('Neodymium', cumulative=True, show = True)
plot_material_demand('Neodymium', cumulative=True, market_share=True, show = True)
plot_material_demand('Neodymium', cumulative=False, market_share=True, show = True)

,2020_2025,2025_2030,2030_2035,2035_2040,2040_2045,2045_2050
Wind_DD-EESG_Onshore,1.410560,0.0,5.057600,0.420907,0.315680,0.167040
Wind_GB-DFIG_SCIG_Onshore,6.045257,0.0,54.477577,5.456754,5.246301,5.296679
Wind_DD-PMSG_Onshore,29.343360,0.0,291.173257,32.387265,34.484282,38.830635
Wind_GB-PMSG_Onshore,6.287863,0.0,66.326811,7.410964,7.922065,9.132952
Sol_C-si_Silver,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
Sol_C-si_Copper,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
Sol_CdTe,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
Sol_CIGS,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
Sol_a-SiGe,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
Hydro,0.000000,0.0,0.000000,0.000000,0.000000,0.000000


# Private Mobility demand for Neodymium

In [9]:
ref_size = {
    'CAR_EV': 54.392,
    'SUV_EV': 54.392,
    'CAR_PHEV': 54.392,
    'SUV_PHEV_GASOLINE': 54.392,
    'COACH_EV': 2152.422,
    'CAR_HEV': 54.392,
    'SUV_HY_GASOLINE': 54.392,
    'BUS_HY_DIESEL': 301.339,
    'SEMI_SH_HY_DIESEL': 322.212,
    'SEMI_LH_HY_DIESEL': 1513.478,
    'LCV_FC_H2' : 33.166,
}

data_vehicles = {}
for period in periods:
    data_vehicles[period] = {
        tech: results['F_new'].loc[period].loc[tech].squeeze()  * 1e6 / ref_size_val
        for tech, ref_size_val in ref_size.items()
    }

df_vehicles = pd.DataFrame(data_vehicles)

In [ ]:
# passer en format long
df_long = df_vehicles.reset_index().melt(
    id_vars='index',
    var_name='Period',
    value_name='Vehicles'
).rename(columns={'index': 'Technology'})

fig = px.bar(
    df_long,
    x='Period',
    y='Vehicles',
    color='Technology',
    title='Number of new electric/hybrid vehicles per period',
)

#filepath = os.path.join(save_dir, 'new_hybrid_elec_vehicles.png')
#fig.write_image(filepath)
fig.show()

In [ ]:
mi_vehicles = {
    'CAR_EV': 695,
    'SUV_EV': 695,
    'CAR_PHEV': 450,
    'SUV_PHEV_GASOLINE': 450,
    'COACH_EV': 1700,
    'CAR_HEV': 278,
    'SUV_HY_GASOLINE': 278,
    'BUS_HY_DIESEL': 278,
    'SEMI_SH_HY_DIESEL': 278,
    'SEMI_LH_HY_DIESEL': 278,
    'LCV_FC_H2': 577,
}

df_demand_vehicles = df_vehicles.multiply(pd.Series(mi_vehicles), axis=0)*0.88 / 1e6

df_melted = df_demand_vehicles.T.reset_index().rename(columns={'index': 'Période'}).melt(
    id_vars='Période', var_name='Technologies', value_name='Demande [t]'
)

fig = px.bar(df_melted, x='Période', y='Demande [t]', color='Technologies', barmode='stack',
             title='Demande en Neodymium - Véhicules')
fig.update_layout(xaxis_title='Période', yaxis_title='Demande [t]')
fig.show()

#filepath = os.path.join(save_dir, 'neo_demand_vehicles.png')
#fig.write_image(filepath)

In [ ]:
def plot_vehicle_demand(cumulative=False, show = True):
    df_demand_vehicles = df_vehicles.multiply(pd.Series(mi_vehicles), axis=0) * 0.88 / 1e6

    if cumulative:
        df_cumulative = df_demand_vehicles.sum().cumsum()
        fig = px.line(x=periods, y=df_cumulative.values,
                      title='Demande cumulative en Neodymium - Véhicules',
                      labels={'x': 'Période', 'y': 'Demande cumulative [t]'})
    else:
        df_melted = df_demand_vehicles.T.reset_index().rename(columns={'index': 'Période'}).melt(
            id_vars='Période', var_name='Technologies', value_name='Demande [t]'
        )
        fig = px.bar(df_melted, x='Période', y='Demande [t]', color='Technologies', barmode='stack',
                     title='Demande en Neodymium - Véhicules')
        fig.update_layout(xaxis_title='Période', yaxis_title='Demande [t]')

    if show:
        fig.show()
        if (cumulative):
            filepath = os.path.join(save_dir, 'neo_demand_vehicle_cumul.png')
        else:
            filepath = os.path.join(save_dir, 'neo_demand_vehicle_no_cumul.png')
        #fig.write_image(filepath)

plot_vehicle_demand(cumulative=False)
plot_vehicle_demand(cumulative=True)


In [ ]:
df_demand_energy = plot_material_demand('Neodymium', cumulative=False, market_share=True, show=False).sum()  # demande énergie par période (sans cumsum)
df_demand_mob = df_demand_vehicles.sum()  # demande mobilité par période

df_total = (df_demand_energy + df_demand_mob).cumsum()

fig = px.line(x=periods, y=df_total.values,
              title='Demande cumulative en Neodymium - Énergie + Mobilité',
              labels={'x': 'Période', 'y': 'Demande cumulative [t]'})

filepath = os.path.join(save_dir, 'neo_demand_energy_vehicle_cumul.png')
#fig.write_image(filepath)
fig.show()

In [ ]:
df_demand_energy = plot_material_demand('Neodymium', cumulative=False, market_share=True).sum()
df_demand_mob = df_demand_vehicles.sum()

df_combined = pd.DataFrame({
    'Énergie': df_demand_energy,
    'Mobilité': df_demand_mob
})

df_melted = df_combined.reset_index().rename(columns={'index': 'Période'}).melt(
    id_vars='Période', var_name='Secteur', value_name='Demande [t]'
)

fig = px.bar(df_melted, x='Période', y='Demande [t]', color='Secteur', barmode='stack',
             title='Demande en Neodymium - Énergie + Mobilité')
fig.update_layout(xaxis_title='Période', yaxis_title='Demande [t]')
fig.show()
filepath = os.path.join(save_dir, 'neo_demand_energy_vehicle.png')
#fig.write_image(filepath)

In [ ]:
years = [2025, 2030, 2035, 2040, 2045, 2050]

df_combined = pd.DataFrame({
    'Énergie': df_demand_energy.values / 5,
    'Mobilité': df_demand_mob.values / 5
}, index=years)

df_combined['Reste de l\'économie'] = (df_combined['Énergie'] + df_combined['Mobilité']) * 70 / 30

df_melted = df_combined.reset_index().rename(columns={'index': 'Année'}).melt(
    id_vars='Année', var_name='Secteur', value_name='Demande [t/an]'
)

fig = px.bar(df_melted, x='Année', y='Demande [t/an]', color='Secteur', barmode='stack',
             title='Demande annuelle en Neodymium - Énergie + Mobilité + Reste')
fig.update_layout(xaxis_title='Année', yaxis_title='Demande [t/an]')
fig.show()

filepath = os.path.join(save_dir, 'neo_demand_energy_vehicle_roe.png')
#fig.write_image(filepath)

In [ ]:
production_2020 = 32898.0177432751
growth_rate = 0.0563770579279044

years_prod = [2025, 2030, 2035, 2040, 2045, 2050]
production = [0.01* production_2020 * (1 + growth_rate) ** (y - 2020) for y in years_prod]

fig = px.line(x=years_prod, y=production,
              title='Neodymium primary refinery capacity',
              labels={'x': 'Année', 'y': 'Production [t/an]'})
fig.show()

filepath = os.path.join(save_dir, 'neo_prod.png')
#fig.write_image(filepath)

In [ ]:
production = [production_2020 * (1 + growth_rate) ** (y - 2020) for y in years_prod]

total_demand = df_combined.sum(axis=1).values  # énergie + mobilité + reste

ratio = total_demand / production * 100

fig = px.line(x=years, y=ratio,
              title='Part de la demande QC sur la production mondiale de Neodymium [%]',
              labels={'x': 'Année', 'y': '% de la production mondiale'})
fig.show()

filepath = os.path.join(save_dir, 'neo_prod_vs_cons.png')
#fig.write_image(filepath)

# Recycling

In [18]:
recycling_rates = {
        2020: 0.15,
        2030: 0.35,
        2040: 0.55,
        2050: 0.75,
    }

In [ ]:
def plot_material_recycled(material, cumulative=False, market_share=False, show = False):
    
    period_recycling = {
        '2020_2025': recycling_rates[2020],
        '2025_2030': (recycling_rates[2020] + recycling_rates[2030]) / 2,
        '2030_2035': recycling_rates[2030],
        '2035_2040': (recycling_rates[2030] + recycling_rates[2040]) / 2,
        '2040_2045': recycling_rates[2040],
        '2045_2050': (recycling_rates[2040] + recycling_rates[2050]) / 2,
    }

    data = {}
    for period in periods:
        data[period] = {}
        rec_rate = period_recycling[period]
        if market_share:
            for model_tech, info in tech_groups.items():
                f_decom = results['F_decom'].xs(model_tech, level='Technologies').loc[period].sum().values[0]
                f_old = results['F_old'].loc[period].loc[model_tech].squeeze()
                f_total = f_decom + f_old
                ms = get_ms(info['iam_source'], period)
                for excel_tech in info['excel_techs']:
                    data[period][excel_tech] = f_total * ms[excel_tech] * df_mi.loc[material, excel_tech] * rec_rate
        else:
            for tech, excel_col in tech_mapping.items():
                f_decom = results['F_decom'].xs(tech, level='Technologies').loc[period].sum().values[0]
                f_old = results['F_old'].loc[period].loc[tech].squeeze()
                f_total = f_decom + f_old
                data[period][tech] = f_total * df_mi.loc[material, excel_col] * rec_rate

    df_plot = pd.DataFrame(data)

    if cumulative:
        df_cumulative = df_plot.sum().cumsum()
        fig = px.line(x=periods, y=df_cumulative.values,
                      title=f'Matériau recyclé cumulatif en {material}',
                      labels={'x': 'Période', 'y': 'Recyclé cumulatif [t]'})
    else:
        df_melted = df_plot.T.reset_index().rename(columns={'index': 'Période'}).melt(
            id_vars='Période', var_name='Technologies', value_name='Recyclé [t]'
        )
        fig = px.bar(df_melted, x='Période', y='Recyclé [t]', color='Technologies', barmode='stack',
                     title=f'Matériau recyclé en {material}')
        fig.update_layout(xaxis_title='Période', yaxis_title='Recyclé [t]')
    if show:
        fig.show()
        if (cumulative and market_share):
            filepath = os.path.join(save_dir, 'neo_recycled_cumul_ms.png')
        if (cumulative == False and market_share == False):
            filepath = os.path.join(save_dir, 'neo_recycled_no_cumul_no_ms.png')
        if (cumulative == True and market_share == False):
            filepath = os.path.join(save_dir, 'neo_recycled_cumul_no_ms.png')
        if (cumulative == False and market_share):
            filepath = os.path.join(save_dir, 'neo_recycled_no_cumul_ms.png')
        #fig.write_image(filepath)
    return df_plot

In [20]:
plot_material_recycled('Neodymium', cumulative=False, market_share=True, show=True)


,2020_2025,2025_2030,2030_2035,2035_2040,2040_2045,2045_2050
Wind_DD-EESG_Onshore,0.199931,0.232376,0.184147,0.189408,0.173624,0.108576
Wind_GB-DFIG_SCIG_Onshore,0.856846,1.422439,1.983523,2.455539,2.885466,3.442842
Wind_DD-PMSG_Onshore,4.159084,7.252184,10.601587,14.574269,18.966355,25.239913
Wind_GB-PMSG_Onshore,0.891232,1.605176,2.414952,3.334934,4.357136,5.936419
Sol_C-si_Silver,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Sol_C-si_Copper,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Sol_CdTe,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Sol_CIGS,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Sol_a-SiGe,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Hydro,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [ ]:
def plot_vehicle_recycled(cumulative=False, show=False):
    period_recycling = {
        '2020_2025': 0.15,
        '2025_2030': 0.25,
        '2030_2035': 0.35,
        '2035_2040': 0.45,
        '2040_2045': 0.55,
        '2045_2050': 0.65,
    }

    data_recycled = {}
    for period in periods:
        rec_rate = period_recycling[period]
        data_recycled[period] = {}
        for tech, ref_size_val in ref_size.items():
            f_decom = results['F_decom'].xs(tech, level='Technologies').loc[period].sum().values[0]
            f_old = results['F_old'].loc[period].loc[tech].squeeze()
            n_vehicles = (f_decom + f_old) * 1e6 / ref_size_val
            data_recycled[period][tech] = n_vehicles * mi_vehicles[tech] * 0.88 * rec_rate / 1e6

    df_recycled = pd.DataFrame(data_recycled)

    if cumulative:
        df_cumulative = df_recycled.sum().cumsum()
        fig = px.line(x=periods, y=df_cumulative.values,
                      title='Neodymium recyclé cumulatif - Véhicules',
                      labels={'x': 'Période', 'y': 'Recyclé cumulatif [t]'})
    else:
        df_melted = df_recycled.T.reset_index().rename(columns={'index': 'Période'}).melt(
            id_vars='Période', var_name='Technologies', value_name='Recyclé [t]'
        )
        fig = px.bar(df_melted, x='Période', y='Recyclé [t]', color='Technologies', barmode='stack',
                     title='Neodymium recyclé - Véhicules')
        fig.update_layout(xaxis_title='Période', yaxis_title='Recyclé [t]')
    if show:
        fig.show()
        if (cumulative):
            filepath = os.path.join(save_dir, 'neo_recyled_vehicle_cumul.png')
        else:
            filepath = os.path.join(save_dir, 'neo_recycled_vehicle_no_cumul.png')
        #fig.write_image(filepath)
    return df_recycled

In [22]:
plot_vehicle_recycled(show=True)

,2020_2025,2025_2030,2030_2035,2035_2040,2040_2045,2045_2050
CAR_EV,1.392378,2.320630,3.248881,11.454753,0.000000,352.252616
SUV_EV,0.506319,0.843865,1.181411,13.243471,214.115596,254.111319
CAR_PHEV,0.682984,1.138307,1.593630,2.930029,0.000000,0.000000
SUV_PHEV_GASOLINE,0.254981,0.424968,0.594955,4.531710,0.000000,0.000000
COACH_EV,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
CAR_HEV,0.483816,0.806360,1.128904,2.198604,0.000000,0.000000
SUV_HY_GASOLINE,0.315043,0.525072,0.735100,3.211243,0.000000,0.000000
BUS_HY_DIESEL,0.005370,0.008949,0.012529,0.028571,0.000000,0.000000
SEMI_SH_HY_DIESEL,0.000000,0.000000,0.000000,0.000000,6.172758,4.460383
SEMI_LH_HY_DIESEL,0.000000,0.000000,0.000000,0.000000,0.000000,1.375510


In [23]:
df_recycled_energy = plot_material_recycled('Neodymium', show=False)
df_recycled_vehicles = plot_vehicle_recycled(show= False)

df_total_recycled = pd.DataFrame({
    'Énergie': df_recycled_energy.sum(),
    'Mobilité': df_recycled_vehicles.sum()
})

df_melted = df_total_recycled.reset_index().rename(columns={'index': 'Période'}).melt(
    id_vars='Période', var_name='Secteur', value_name='Recyclé [t]'
)

fig = px.bar(df_melted, x='Période', y='Recyclé [t]', color='Secteur', barmode='stack',
             title='Neodymium recyclé - Énergie + Mobilité')
fig.update_layout(xaxis_title='Période', yaxis_title='Recyclé [t]')
fig.show()

In [24]:
df_total_recycled = pd.DataFrame({
    'Énergie': df_recycled_energy.sum().values / 5,
    'Mobilité': df_recycled_vehicles.sum().values / 5
}, index=years)

df_melted = df_total_recycled.reset_index().rename(columns={'index': 'Année'}).melt(
    id_vars='Année', var_name='Secteur', value_name='Recyclé [t/an]'
)

fig = px.bar(df_melted, x='Année', y='Recyclé [t/an]', color='Secteur', barmode='stack',
             title='Neodymium recyclé annuel - Énergie + Mobilité')
fig.update_layout(xaxis_title='Année', yaxis_title='Recyclé [t/an]')
fig.show()

In [ ]:
df_demand_total = pd.DataFrame({
    'Énergie': df_demand_energy.values / 5,
    'Mobilité': df_demand_mob.values / 5
}, index=years)

gross = df_demand_total.sum(axis=1)
recycled = df_total_recycled.sum(axis=1)
net = gross - recycled

df_melted_demand = df_demand_total.reset_index().rename(columns={'index': 'Année'}).melt(
    id_vars='Année', var_name='Secteur', value_name='Demande [t/an]'
)

fig = px.bar(df_melted_demand, x='Année', y='Demande [t/an]', color='Secteur', barmode='stack',
             title='Demande en Neodymium - Brute vs Nette')

fig.add_scatter(
    x=years, y=net.values,
    mode='markers',
    marker=dict(symbol='triangle-up', size=12, color='black'),
    name='Demande nette'
)

fig.update_layout(xaxis_title='Année', yaxis_title='[t/an]')
fig.show()

filepath = os.path.join(save_dir, 'neo_net_demand.png')
#fig.write_image(filepath)

In [32]:
df_demand_total 

,Énergie,Mobilité
2025,8.617408,16.710392
2030,0.000000,106.375214
2035,83.407049,202.113705
2040,9.135178,174.740982
2045,9.593666,209.251171
2050,10.685461,201.160286


# TECHNOLOGIES

In [36]:
keywords_elec = ['PV_', 'WIND_', 'HYDRO', 'NUCLEAR', 'CCGT', 'COAL_', 'OCGT_', 'TIDAL', 'GEOTHERMAL', 'AFC', 'PAFC', 'PEMFC', 'SOFC', 'WAVE']

keywords_public = ['BUS_', 'TRAMWAY', 'SCHOOLBUS_', 'COMMUTER_RAIL', 'COACH_','PLANE_SH', 'PLANE_LH', 'TRAIN_DIESEL', 'TRAIN_BIODIESEL','TRAIN_ELEC', 'TRAIN_H2', 'TRAIN_NG', 'TRAIN_SNG']

keywords_freight =  ['TRUCK_', 'SEMI_', 'LCV_', 'TRAIN_FREIGHT', 'BULK_CARRIER', 'CONTAINER', 'OIL_TANKER', 'PLANE_FREIGHT']

keywords_priv = ['CAR_', 'SUV_']
 

In [37]:
elec_techs = [t for t in results['F_new'].loc['2020_2025'].index 
              if any(kw in t for kw in keywords_elec) and not t.startswith('COAL_GAS')]

In [29]:
pd.DataFrame(elec_techs).to_excel('technologies_elec.xlsx', index=False)

In [ ]:
all_techs = [t for t in results['F_new'].loc['2020_2025'].index]

pd.DataFrame(all_techs).to_excel('all_technologies.xlsx', index=False)

In [ ]:
priv_mob_techs = [t for t in results['F_new'].loc['2020_2025'].index 
              if any(kw in t for kw in keywords_priv) and not t.endswith(('_ELD', '_MD', '_SD', 'LD'))]


pd.DataFrame(priv_mob_techs).to_excel('priv_mob_technologies.xlsx', index=False)

In [38]:
pub_mob_techs = [t for t in results['F_new'].loc['2020_2025'].index 
              if any(kw in t for kw in keywords_public) and not t.endswith(('_ELD', '_MD', '_SD', 'LD'))]


pd.DataFrame(pub_mob_techs).to_excel('pub_mob_technologies.xlsx', index=False)

In [39]:
freight_techs = [t for t in results['F_new'].loc['2020_2025'].index 
              if any(kw in t for kw in keywords_freight)]


pd.DataFrame(freight_techs).to_excel('freight_technologies.xlsx', index=False)